# 04 — Train frontal-eddy U-Net (Colab GPU)

Segmentation: (SSHA, ADT, |∇ADT|) 128×128 tiles → frontal-eddy mask. Targets = PET
speed contours of hand-confirmed frontal eddies (`scripts/build_masks.py`).

Temporal split: **train 1–30** (fit) / **val 31–35** (model selection) / **cal 36–42**
(conformal calibration, untouched by any modeling choice) / **test 43–50** (final eval).

**Colab setup:** zip `out/aoi5/training/` locally → upload to Drive. Repo code comes
from GitHub clone. Known v1 caveats: PET-missed frontals are background in targets;
per-tile inpaint seams in inputs.

In [ ]:
# SELF-CONTAINED Colab trainer — needs only training data at /content/training
# (unzip cell below) and a GPU runtime. Saves weights + conformal threshold to Drive.
from google.colab import drive
drive.mount("/content/drive")

import zipfile, os
from pathlib import Path
if not Path("/content/training/training_manifest.csv").exists():
    with zipfile.ZipFile("/content/drive/MyDrive/swot-eddy-ml/training.zip") as z:
        z.extractall("/content")
TRAIN_DIR = "/content/training"
OUT_DIR = "/content/drive/MyDrive/swot-eddy-ml/model"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

import torch, numpy as np
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEV, "| files:", len(list(Path(TRAIN_DIR).glob('*.npz'))))


In [ ]:
# ---- model + dataset (inlined from src/swot_eddy/unet.py) ----
"""Small U-Net for frontal-eddy segmentation on SWOT swaths, plus the tile dataset.

Torch imports are kept inside functions/classes so the rest of the package works
without torch installed (torch is only needed for training/inference).

Input : (3, 128, 128) tiles — inpainted SSHA [m], inpainted ADT [m], |grad ADT| [m/km]
Target: (1, 128, 128) binary mask (1 = inside a confirmed frontal eddy's PET contour)
Built from the per-pass .npz files produced by scripts/build_masks.py.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np

# Channels are first CLIPPED to physical bounds, then standardized. The clip is
# load-bearing: biharmonic inpainting explodes inside large gaps (ADT p99 was +120 m
# before clipping) and ssha_unedited carries raw outliers; bounds turn blowups into
# flat "no-data-like" patches. Stats measured on the clipped training distribution
# (cycles 1-35 sample) and FROZEN so inference scales identically.
CHANNEL_CLIP = np.array([[-1.0, 1.5], [-0.6, 1.6], [0.0, 0.05]], dtype=np.float32)
CHANNEL_MEAN = np.array([0.167, 0.606, 0.0089], dtype=np.float32)
CHANNEL_STD = np.array([0.234, 0.582, 0.0146], dtype=np.float32)
TILE = 128


def build_unet(base: int = 32):
    """3-level U-Net (~1.9M params at base=32). Returns nn.Module: (B,3,128,128)->(B,1,128,128) logits."""
    import torch
    import torch.nn as nn

    def block(ci, co):
        return nn.Sequential(
            nn.Conv2d(ci, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
            nn.Conv2d(co, co, 3, padding=1, bias=False), nn.BatchNorm2d(co), nn.ReLU(inplace=True),
        )

    class UNet(nn.Module):
        def __init__(self):
            super().__init__()
            b = base
            self.e1, self.e2, self.e3 = block(3, b), block(b, 2 * b), block(2 * b, 4 * b)
            self.mid = block(4 * b, 8 * b)
            self.pool = nn.MaxPool2d(2)
            self.u3 = nn.ConvTranspose2d(8 * b, 4 * b, 2, stride=2)
            self.d3 = block(8 * b, 4 * b)
            self.u2 = nn.ConvTranspose2d(4 * b, 2 * b, 2, stride=2)
            self.d2 = block(4 * b, 2 * b)
            self.u1 = nn.ConvTranspose2d(2 * b, b, 2, stride=2)
            self.d1 = block(2 * b, b)
            self.head = nn.Conv2d(b, 1, 1)

        def forward(self, x):
            import torch
            s1 = self.e1(x)
            s2 = self.e2(self.pool(s1))
            s3 = self.e3(self.pool(s2))
            m = self.mid(self.pool(s3))
            d = self.d3(torch.cat([self.u3(m), s3], 1))
            d = self.d2(torch.cat([self.u2(d), s2], 1))
            d = self.d1(torch.cat([self.u1(d), s1], 1))
            return self.head(d)

    return UNet()


def normalize(x: np.ndarray) -> np.ndarray:
    """(3,H,W) float -> clip to physical bounds, standardize, NaN -> 0."""
    x = np.clip(x.astype(np.float32),
                CHANNEL_CLIP[:, 0, None, None], CHANNEL_CLIP[:, 1, None, None])
    out = (x - CHANNEL_MEAN[:, None, None]) / CHANNEL_STD[:, None, None]
    return np.nan_to_num(out, nan=0.0)


def pass_tiles(x, y=None, stride: int = 64):
    """Cut a stitched pass (3,L,128) [+ mask (L,128)] into (start, x_tile[, y_tile]) 128-tiles,
    same tiling scheme as detection (final window snapped to the end)."""
    L = x.shape[1]
    starts = list(range(0, max(L - TILE, 0) + 1, stride)) or [0]
    if starts[-1] != max(L - TILE, 0):
        starts.append(max(L - TILE, 0))
    out = []
    for s in starts:
        xt = x[:, s:s + TILE, :]
        if xt.shape[1] < TILE:
            xt = np.pad(xt, ((0, 0), (0, TILE - xt.shape[1]), (0, 0)))
        if y is None:
            out.append((s, xt))
        else:
            yt = y[s:s + TILE, :]
            if yt.shape[0] < TILE:
                yt = np.pad(yt, ((0, TILE - yt.shape[0]), (0, 0)))
            out.append((s, xt, yt))
    return out


class TileDataset:
    """Torch dataset over build_masks .npz files for one split.

    pos_frac: fraction of samples guaranteed to contain mask pixels (positive tiles are
    rare — ~1 in 8 — so training batches are rebalanced; validation uses pos_frac=None
    = natural distribution, every tile enumerated once).
    """

    def __init__(self, training_dir, split, pos_frac=0.5, seed=0, augment=False):
        import pandas as pd
        self.dir = Path(training_dir)
        man = pd.read_csv(self.dir / "training_manifest.csv", dtype={"cycle": str, "pass": str})
        man = man[(man.split == split) & (man.status == "ok")]
        self.files = [self.dir / f"{c}_{p}.npz" for c, p in zip(man.cycle, man["pass"])]
        self.files = [f for f in self.files if f.exists()]
        self.pos_frac = pos_frac
        self.augment = augment
        self.rng = np.random.default_rng(seed)
        self.index = []          # (file_i, start, has_pos)
        for i, f in enumerate(self.files):
            with np.load(f) as d:
                y = d["y"]
                for s, _, yt in pass_tiles(np.zeros((3,) + y.shape, np.float16), y):
                    self.index.append((i, s, bool(yt.any())))
        self.pos = [t for t in self.index if t[2]]
        self.neg = [t for t in self.index if not t[2]]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, k):
        import torch
        if self.pos_frac is not None:
            pool = self.pos if self.rng.random() < self.pos_frac and self.pos else self.neg
            i, s, _ = pool[self.rng.integers(len(pool))]
        else:
            i, s, _ = self.index[k]
        with np.load(self.files[i]) as d:
            x = d["x"][:, s:s + TILE, :].astype(np.float32)
            y = d["y"][s:s + TILE, :].astype(np.float32)
        if x.shape[1] < TILE:
            x = np.pad(x, ((0, 0), (0, TILE - x.shape[1]), (0, 0)))
            y = np.pad(y, ((0, TILE - y.shape[0]), (0, 0)))
        x = normalize(x)
        if self.augment:
            if self.rng.random() < 0.5:          # along-track flip
                x, y = x[:, ::-1, :].copy(), y[::-1, :].copy()
            if self.rng.random() < 0.5:          # across-track flip
                x, y = x[:, :, ::-1].copy(), y[:, ::-1].copy()
        return torch.from_numpy(x), torch.from_numpy(y[None])


def dice_bce_loss(logits, target, bce_weight: float = 0.5):
    import torch
    import torch.nn.functional as F
    bce = F.binary_cross_entropy_with_logits(logits, target)
    p = torch.sigmoid(logits)
    inter = (p * target).sum(dim=(1, 2, 3))
    dice = 1 - (2 * inter + 1) / (p.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + 1)
    return bce_weight * bce + (1 - bce_weight) * dice.mean()


In [ ]:
from torch.utils.data import DataLoader

train_ds = TileDataset(TRAIN_DIR, "train", pos_frac=0.5, augment=True)
val_ds = TileDataset(TRAIN_DIR, "val", pos_frac=None)
print(f"train: {len(train_ds)} tiles ({len(train_ds.pos)} positive)")
print(f"val:   {len(val_ds)} tiles ({len(val_ds.pos)} positive)")
train_dl = DataLoader(train_ds, batch_size=32, num_workers=2, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32, num_workers=2)

In [ ]:
def val_metrics(model):
    model.eval()
    inter = union = tp = fp = fn = 0
    with torch.no_grad():
        for x, y in val_dl:
            p = (torch.sigmoid(model(x.to(DEV))) > 0.5).float().cpu()
            inter += (p * y).sum().item(); union += ((p + y) > 0).sum().item()
            tp += ((p == 1) & (y == 1)).sum().item()
            fp += ((p == 1) & (y == 0)).sum().item()
            fn += ((p == 0) & (y == 1)).sum().item()
    iou = inter / max(union, 1)
    return iou, tp / max(tp + fp, 1), tp / max(tp + fn, 1)

model = build_unet().to(DEV)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=30)
best_iou, EPOCHS = 0.0, 30
for ep in range(EPOCHS):
    model.train()
    tot = 0.0
    for x, y in train_dl:
        opt.zero_grad()
        loss = dice_bce_loss(model(x.to(DEV)), y.to(DEV))
        loss.backward(); opt.step()
        tot += loss.item()
    sched.step()
    iou, prec, rec = val_metrics(model)
    star = ""
    if iou > best_iou:
        best_iou = iou
        torch.save(model.state_dict(), f"{OUT_DIR}/unet_frontal.pt")
        star = " *saved"
    print(f"ep {ep+1:02d}  loss {tot/len(train_dl):.4f}  "
          f"val IoU {iou:.3f} P {prec:.3f} R {rec:.3f}{star}")

In [ ]:
# eddy-LEVEL evaluation on the held-out TEST split (cycles 43-50): a predicted blob
# counts as a hit if it overlaps a true frontal mask (IoU-free, detection-style).
from scipy import ndimage
import pandas as pd

model.load_state_dict(torch.load(f"{OUT_DIR}/unet_frontal.pt", map_location=DEV))
model.eval()
man = pd.read_csv(f"{TRAIN_DIR}/training_manifest.csv", dtype={"cycle": str, "pass": str})
test = man[(man.split == "test") & (man.status == "ok")]
TP = FP = FN = 0
for c, p in zip(test.cycle, test["pass"]):
    with np.load(f"{TRAIN_DIR}/{c}_{p}.npz") as d:
        x, y = d["x"].astype(np.float32), d["y"]
    L = x.shape[1]
    prob = np.zeros((L, 128), np.float32); hits = np.zeros((L, 128), np.float32)
    with torch.no_grad():
        for s, xt in pass_tiles(x):
            pr = torch.sigmoid(model(torch.from_numpy(normalize(xt))[None].to(DEV)))[0, 0].cpu().numpy()
            e = min(s + TILE, L)
            prob[s:e] += pr[: e - s]; hits[s:e] += 1
    pred = (prob / np.maximum(hits, 1)) > 0.5
    pl, pn = ndimage.label(pred)
    tl, tn = ndimage.label(y)
    hit_true = set(np.unique(tl[pred])) - {0}
    hit_pred = set(np.unique(pl[y > 0])) - {0}
    TP += len(hit_true); FN += tn - len(hit_true); FP += pn - len(hit_pred)
print(f"TEST eddy-level: precision {TP/max(TP+FP,1):.2f}  recall {TP/max(TP+FN,1):.2f}  "
      f"(TP {TP} FP {FP} FN {FN})")
print("NOTE: recall is vs PET-derived truth — PET-missed frontals aren't counted.")

In [ ]:
# CONFORMAL calibration (cal split, cycles 36-42): pick detection threshold lam_hat with
# a distribution-free guarantee that a new frontal eddy is missed with prob <= ALPHA.
# Score per true eddy = max predicted probability inside its mask. lam_hat = the
# floor(ALPHA*(n+1)-1)/n lower empirical quantile of the scores (split-conformal bound).
# Caveat (report honestly): guarantee assumes eddy exchangeability; eddies from the same
# cycle are weakly dependent, so coverage is approximate at the cycle level.
ALPHA = 0.10

def eddy_scores(split):
    scores = []
    sub = man[(man.split == split) & (man.status == "ok") & (man.n_frontal > 0)]
    for c, p in zip(sub.cycle, sub["pass"]):
        with np.load(f"{TRAIN_DIR}/{c}_{p}.npz") as d:
            x, y = d["x"].astype(np.float32), d["y"]
        L = x.shape[1]
        prob = np.zeros((L, 128), np.float32); hits = np.zeros((L, 128), np.float32)
        with torch.no_grad():
            for s, xt in pass_tiles(x):
                pr = torch.sigmoid(model(torch.from_numpy(normalize(xt))[None].to(DEV)))[0, 0].cpu().numpy()
                e = min(s + TILE, L)
                prob[s:e] += pr[: e - s]; hits[s:e] += 1
        prob /= np.maximum(hits, 1)
        tl, tn = ndimage.label(y)
        for k in range(1, tn + 1):
            scores.append(float(prob[tl == k].max()))
    return np.array(scores)

cal_scores = eddy_scores("cal")
n = len(cal_scores)
k = int(np.floor(ALPHA * (n + 1) - 1))
lam_hat = float(np.sort(cal_scores)[max(k, 0)])
print(f"cal eddies: {n} | lam_hat = {lam_hat:.3f} (target miss rate <= {ALPHA:.0%})")

# empirical check on TEST: fraction of true frontal eddies with max-prob >= lam_hat
test_scores = eddy_scores("test")
print(f"test coverage: {np.mean(test_scores >= lam_hat):.1%} of {len(test_scores)} "
      f"frontal eddies detected at lam_hat (guarantee: >= {1-ALPHA:.0%})")
np.save(f"{OUT_DIR}/lam_hat.npy", np.array([lam_hat, ALPHA]))